In [ ]:
!pip install roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="JTCh2M1hSr8zMo9grmxU")

# 기존 데이터셋
project1 = rf.workspace("gerard-hernandez-i2d5h").project("head-detection-pyxh4")
version1 = project1.version(1)
dataset1 = version1.download("yolov8")

# 새 데이터셋
project2 = rf.workspace("thai-food-9tgjo").project("head-detection-iry9q")
version2 = project2.version(6)
dataset2 = version2.download("yolov8")

print("완료!")
print(f"dataset1 위치: {dataset1.location}")
print(f"dataset2 위치: {dataset2.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Head-Detection-1 in yolov8:: 100%|██████████| 4670/4670 [00:00<00:00, 5260.04it/s]

loading Roboflow workspace...


loading Roboflow project...
완료!
dataset1 위치: /content/Head-Detection-1
dataset2 위치: /content/Head-Detection-6


In [ ]:
import os
import shutil

os.makedirs('/content/merged/train/images', exist_ok=True)
os.makedirs('/content/merged/train/labels', exist_ok=True)
os.makedirs('/content/merged/valid/images', exist_ok=True)
os.makedirs('/content/merged/valid/labels', exist_ok=True)

def copy_files(img_src, lbl_src, split, prefix):
    if not os.path.exists(img_src):
        print(f"{img_src} 없음 스킵")
        return
    for f in os.listdir(img_src):
        shutil.copy(os.path.join(img_src, f), f'/content/merged/{split}/images/{prefix}_{f}')
    for f in os.listdir(lbl_src):
        shutil.copy(os.path.join(lbl_src, f), f'/content/merged/{split}/labels/{prefix}_{f}')

# 기존 데이터 복사
copy_files(f'{dataset1.location}/train/images',
           f'{dataset1.location}/train/labels', 'train', 'ds1')
copy_files(f'{dataset1.location}/valid/images',
           f'{dataset1.location}/valid/labels', 'valid', 'ds1')

# 새 데이터 복사
copy_files(f'{dataset2.location}/train/images',
           f'{dataset2.location}/train/labels', 'train', 'ds2')
copy_files(f'{dataset2.location}/valid/images',
           f'{dataset2.location}/valid/labels', 'valid', 'ds2')

print("합치기 완료!")
print(f"Train 이미지: {len(os.listdir('/content/merged/train/images'))}장")
print(f"Valid 이미지: {len(os.listdir('/content/merged/valid/images'))}장")

합치기 완료!
Train 이미지: 5298장
Valid 이미지: 615장


In [ ]:
!pip install ultralytics

yaml_content = """train: /content/merged/train/images
val: /content/merged/valid/images

nc: 1
names: ['head']
"""

with open('/content/merged/data.yaml', 'w') as f:
    f.write(yaml_content)

from ultralytics import YOLO
import shutil, os

model = YOLO('yolov8n.pt')
model.train(
    data='/content/merged/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='merged_head_model'
)

# 학습 끝나자마자 바로 Drive에 저장!
os.makedirs('/content/drive/MyDrive/head_project_merged', exist_ok=True)
shutil.copy(
    '/content/runs/detect/merged_head_model/weights/best.pt',
    '/content/drive/MyDrive/head_project_merged/best.pt'
)
print("Drive 저장 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

In [ ]:
import os

# 기존 데이터셋
ds1_train = len(os.listdir(f'{dataset1.location}/train/images'))
ds1_valid = len(os.listdir(f'{dataset1.location}/valid/images'))

# 새 데이터셋
ds2_train = len(os.listdir(f'{dataset2.location}/train/images'))
ds2_valid = len(os.listdir(f'{dataset2.location}/valid/images'))

print(f"기존 데이터셋 - Train: {ds1_train}장, Valid: {ds1_valid}장, 합계: {ds1_train+ds1_valid}장")
print(f"새 데이터셋  - Train: {ds2_train}장, Valid: {ds2_valid}장, 합계: {ds2_train+ds2_valid}장")
print(f"총 합계: {ds1_train+ds1_valid+ds2_train+ds2_valid}장")

기존 데이터셋 - Train: 1629장, Valid: 462장, 합계: 2091장
새 데이터셋  - Train: 3669장, Valid: 153장, 합계: 3822장
총 합계: 5913장


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/head_project_merged/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>